# 🤟 Training Model Klasifikasi Abjad Bahasa Isyarat SIBI / BISINDO
Notebook ini melatih model Machine Learning untuk mengenali 26 huruf Abjad Bahasa Isyarat (A-Z) menggunakan koordinat **21 Landmark Sendi Jari MediaPipe**.

### Alur Pelatihan:
1. **Ekstraksi Fitur**: Menemukan 21 sendi tangan dan menormalisasi koordinatnya (bebas dari pengaruh posisi/jarak tangan ke kamera).
2. **Dataset**: Menggunakan data dari `dataset/sibi/abjad` atau `dataset/bisindo - video dataset`.
3. **Training Model**: Melatih model **Random Forest Classifier** & **MLP (Multi-Layer Perceptron)**.
4. **Evaluasi**: Mengukur akurasi, Confusion Matrix, dan Classification Report.
5. **Ekspor Model**: Menyimpan model ke `model/sibi_abjad_model.pkl` dan `model/sibi_labels.json` agar langsung terintegrasi dengan web server dashboard.

### 1. Import Library & Persiapan Lingkungan

In [5]:
import os
import re
import json
import math
import pickle
import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix

# Periksa MediaPipe
import mediapipe as mp
print("✓ OpenCV Version:", cv2.__version__)
print("✓ MediaPipe Version:", mp.__version__)

✓ OpenCV Version: 5.0.0
✓ MediaPipe Version: 1.1.1


### 2. Fungsi Normalisasi Landmark (63 Fitur Invariant Skala & Translasi)
Rumus ini penting: 
- **Translasi**: Titik pergelangan tangan (wrist / index 0) dijadikan acuan `(0, 0, 0)`.
- **Skala**: Seluruh koordinat dibagi dengan jarak pergelangan tangan ke pangkal jari tengah (index 9).

In [6]:
def extract_normalized_features_from_coords(coords_21):
    """
    coords_21: list of (x, y, z) 21 titik sendi tangan
    Menghasilkan 63 float ternormalisasi.
    """
    if len(coords_21) < 21:
        return None

    wrist_x, wrist_y, wrist_z = coords_21[0]
    ref_x = coords_21[9][0] - wrist_x
    ref_y = coords_21[9][1] - wrist_y
    ref_z = coords_21[9][2] - wrist_z
    scale = math.sqrt(ref_x**2 + ref_y**2 + ref_z**2)
    if scale == 0:
        scale = 1.0

    features = []
    for pt in coords_21:
        rel_x = (pt[0] - wrist_x) / scale
        rel_y = (pt[1] - wrist_y) / scale
        rel_z = (pt[2] - wrist_z) / scale
        features.extend([rel_x, rel_y, rel_z])

    return features

### 3. Inisialisasi Hand Landmarker MediaPipe

In [7]:
MODEL_TASK_PATH = "../model/hand_landmarker.task"

# MediaPipe Tasks Vision Setup
from mediapipe.tasks import python
from mediapipe.tasks.python import vision

base_options = python.BaseOptions(model_asset_path=MODEL_TASK_PATH)
options = vision.HandLandmarkerOptions(
    base_options=base_options,
    num_hands=1,
    min_hand_detection_confidence=0.3,
    min_hand_presence_confidence=0.3
)
detector = vision.HandLandmarker.create_from_options(options)
print("✓ Hand Landmarker berhasil diinisialisasi.")

W0000 00:00:1793070192.264368    1884 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1793070192.304001    1884 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.


✓ Hand Landmarker berhasil diinisialisasi.


### 4. Ekstraksi Fitur dari Folder Dataset Citra / Video
Membaca citra abjad dari folder dataset dan mengekstrak koordinatnya.

In [8]:
def extract_features_from_image(image_path):
    img = cv2.imread(image_path)
    if img is None:
        return None
    rgb_img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    mp_img = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb_img)

    try:
        res = detector.detect(mp_img)
        if res and res.hand_landmarks:
            raw_lms = [(lm.x, lm.y, lm.z) for lm in res.hand_landmarks[0]]
            return extract_normalized_features_from_coords(raw_lms)
    except Exception:
        pass
    return None

X_data = []
y_labels = []

# Prioritas direktori dataset alfabet SIBI
CANDIDATE_DIRS = [
    "../dataset/SIBI Indonesian Sign Language Alphabet Image Datas/SIBI/Mono_Background",
    "../dataset/sibi/abjad",
    "../dataset/SIBI Indonesian Sign Language Alphabet Image Datas"
]

DATASET_DIR = next((d for d in CANDIDATE_DIRS if os.path.exists(d)), None)

if DATASET_DIR:
    print(f"[+] Mengekstrak fitur dari dataset: {DATASET_DIR}")
    classes = sorted([d for d in os.listdir(DATASET_DIR) if os.path.isdir(os.path.join(DATASET_DIR, d))])
    print(f"[+] Ditemukan {len(classes)} kelas alfabet: {classes}")
    for class_name in classes:
        class_dir = os.path.join(DATASET_DIR, class_name)
        image_files = [f for f in os.listdir(class_dir) if f.lower().endswith((".png", ".jpg", ".jpeg"))]
        extracted_count = 0
        for f in image_files:
            f_path = os.path.join(class_dir, f)
            feats = extract_features_from_image(f_path)
            if feats is not None:
                X_data.append(feats)
                y_labels.append(class_name.upper())
                extracted_count += 1
        print(f"    - Kelas {class_name.upper()}: {extracted_count}/{len(image_files)} gambar berhasil dideteksi")
else:
    print("[!] Dataset tidak ditemukan pada kandidat path yang ditentukan!")

print(f"Total sampel landmark terdeteksi: {len(X_data)}")
if len(X_data) > 0:
    print("Distribusi kelas:", pd.Series(y_labels).value_counts())


[+] Mengekstrak fitur dari dataset: ../dataset/SIBI Indonesian Sign Language Alphabet Image Datas/SIBI/Mono_Background
[+] Ditemukan 24 kelas alfabet: ['A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y']


W0000 00:00:1793070192.768150    1886 landmark_projection_calculator.cc:81] Using NORM_RECT without IMAGE_DIMENSIONS is only supported for the square ROI. Provide IMAGE_DIMENSIONS or use PROJECTION_MATRIX.


    - Kelas A: 59/60 gambar berhasil dideteksi
    - Kelas B: 60/60 gambar berhasil dideteksi
    - Kelas C: 60/60 gambar berhasil dideteksi
    - Kelas D: 59/60 gambar berhasil dideteksi
    - Kelas E: 60/60 gambar berhasil dideteksi
    - Kelas F: 60/60 gambar berhasil dideteksi
    - Kelas G: 60/60 gambar berhasil dideteksi
    - Kelas H: 60/60 gambar berhasil dideteksi
    - Kelas I: 60/60 gambar berhasil dideteksi
    - Kelas K: 60/60 gambar berhasil dideteksi
    - Kelas L: 60/60 gambar berhasil dideteksi
    - Kelas M: 60/60 gambar berhasil dideteksi
    - Kelas N: 60/60 gambar berhasil dideteksi
    - Kelas O: 59/60 gambar berhasil dideteksi
    - Kelas P: 54/60 gambar berhasil dideteksi
    - Kelas Q: 60/60 gambar berhasil dideteksi
    - Kelas R: 60/60 gambar berhasil dideteksi
    - Kelas S: 59/60 gambar berhasil dideteksi
    - Kelas T: 60/60 gambar berhasil dideteksi
    - Kelas U: 60/60 gambar berhasil dideteksi
    - Kelas V: 60/60 gambar berhasil dideteksi
    - Kelas W

### 5. (Opsional) Rekam Sampel Tambahan Langsung dari Webcam
Jika data citra masih sedikit, kamu bisa menjalankan cell ini untuk merekam variasi pose tanganmu sendiri secara langsung dari webcam!

In [9]:
# Set HURUF yang ingin direkam dan jumlah frame (misal 50 sampel)
REKAM_HURUF = "A"
JUMLAH_SAMPEL = 40

def rekam_dari_webcam(huruf_target, target_count=40):
    cap = cv2.VideoCapture(0)
    samples = []
    print(f"[!] Siapkan pose tangan untuk huruf: '{huruf_target}'. Tekan 'q' untuk keluar.")

    while cap.isOpened() and len(samples) < target_count:
        ret, frame = cap.read()
        if not ret:
            break
        rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        mp_img = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb)
        res = detector.detect(mp_img)

        if res and res.hand_landmarks:
            raw_lms = [(lm.x, lm.y, lm.z) for lm in res.hand_landmarks[0]]
            feats = extract_normalized_features_from_coords(raw_lms)
            if feats:
                samples.append(feats)
                cv2.putText(frame, f"Merekam: {len(samples)}/{target_count}", (20, 50),
                            cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)

        cv2.imshow("Perekam Pose Tangan SIBI", frame)
        if cv2.waitKey(100) & 0xFF == ord('q'):
            break

    cap.release()
    cv2.destroyAllWindows()
    return samples

# Jalankan jika ingin merekam: (hapus komentar di bawah jika ingin digunakan)
# extra_samples = rekam_dari_webcam(REKAM_HURUF, JUMLAH_SAMPEL)
# X_data.extend(extra_samples)
# y_labels.extend([REKAM_HURUF] * len(extra_samples))

### 6. Data Augmentasi (Jitter & Rotasi Ringan)
Menambahkan sedikit variasi noise matematis pada koordinat sendi agar model lebih tangguh terhadap getaran tangan.

In [10]:
def augment_features(features_list, labels_list, factor=10):
    aug_X = []
    aug_y = []
    for feats, label in zip(features_list, labels_list):
        aug_X.append(feats)
        aug_y.append(label)
        for _ in range(factor):
            noise = np.random.normal(0, 0.015, len(feats))
            aug_X.append(list(np.array(feats) + noise))
            aug_y.append(label)
    return aug_X, aug_y

if len(X_data) > 0:
    X_augmented, y_augmented = augment_features(X_data, y_labels, factor=15)
    print(f"Total data setelah augmentasi: {len(X_augmented)}")
else:
    print("[!] Unduh dataset SIBI terlebih dahulu menggunakan: python3 scrape_sibi.py --category abjad")

Total data setelah augmentasi: 22880


### 7. Training Model Classifier (Random Forest & MLP)

In [11]:
if len(X_data) > 0:
    X_train, X_test, y_train, y_test = train_test_split(
        X_augmented, y_augmented, test_size=0.2, random_state=42, stratify=y_augmented
    )

    print(f"Data Train: {len(X_train)} | Data Test: {len(X_test)}")

    # Model 1: Random Forest Classifier
    rf_model = RandomForestClassifier(n_estimators=100, max_depth=15, random_state=42)
    rf_model.fit(X_train, y_train)

    y_pred_rf = rf_model.predict(X_test)
    acc_rf = accuracy_score(y_test, y_pred_rf)
    print(f"\n✓ Akurasi Random Forest: {acc_rf * 100:.2f}%")

    # Model 2: Multi-Layer Perceptron (Neural Network)
    mlp_model = MLPClassifier(hidden_layer_sizes=(128, 64), max_iter=300, random_state=42)
    mlp_model.fit(X_train, y_train)

    y_pred_mlp = mlp_model.predict(X_test)
    acc_mlp = accuracy_score(y_test, y_pred_mlp)
    print(f"✓ Akurasi MLP Neural Network: {acc_mlp * 100:.2f}%")

    best_model = rf_model if acc_rf >= acc_mlp else mlp_model
    print("\n--- Classification Report ---")
    print(classification_report(y_test, y_pred_rf))

Data Train: 18304 | Data Test: 4576

✓ Akurasi Random Forest: 100.00%
✓ Akurasi MLP Neural Network: 100.00%

--- Classification Report ---
              precision    recall  f1-score   support

           A       1.00      1.00      1.00       189
           B       1.00      1.00      1.00       192
           C       1.00      1.00      1.00       192
           D       1.00      1.00      1.00       188
           E       1.00      1.00      1.00       192
           F       1.00      1.00      1.00       192
           G       1.00      1.00      1.00       192
           H       1.00      1.00      1.00       192
           I       1.00      1.00      1.00       192
           K       1.00      1.00      1.00       192
           L       1.00      1.00      1.00       192
           M       1.00      1.00      1.00       192
           N       1.00      1.00      1.00       192
           O       1.00      1.00      1.00       189
           P       1.00      1.00      1.00       

### 8. Simpan Model Terlatih ke `model/`
Model akan disimpan ke format `.pkl` dan label kelas ke `.json` agar langsung dikenali oleh web server penerjemah SIBI.

In [12]:
OUTPUT_MODEL_DIR = "../model"
os.makedirs(OUTPUT_MODEL_DIR, exist_ok=True)

model_save_path = os.path.join(OUTPUT_MODEL_DIR, "sibi_abjad_model.pkl")
labels_save_path = os.path.join(OUTPUT_MODEL_DIR, "sibi_labels.json")

if 'best_model' in locals():
    # Simpan model
    with open(model_save_path, "wb") as f:
        pickle.dump(best_model, f)

    # Simpan label kelas
    classes = list(best_model.classes_)
    with open(labels_save_path, "w", encoding="utf-8") as f:
        json.dump(classes, f, indent=2)

    print(f"[✓] Model berhasil diekspor:")
    print(f"    - Model file : {model_save_path}")
    print(f"    - Labels file: {labels_save_path} ({len(classes)} kelas)")
    print("\nBuka tab 'Penerjemah SIBI' di browser untuk langsung mencoba hasil terjemahan!")
else:
    print("[!] Jalankan training model terlebih dahulu sebelum menyimpan.")

[✓] Model berhasil diekspor:
    - Model file : ../model/sibi_abjad_model.pkl
    - Labels file: ../model/sibi_labels.json (24 kelas)

Buka tab 'Penerjemah SIBI' di browser untuk langsung mencoba hasil terjemahan!
